In [ ]:
import os
import datetime, time
import json
from io import BytesIO
from zipfile import ZipFile
import pandas as pd
from pathlib import Path
import requests
from pandas import json_normalize
import re
from urllib.request import urlopen 
import fnmatch  
import xml.etree.ElementTree as et
from math import floor
from github import Github
from github.GithubException import BadCredentialsException
#from collect_commits import extract_commits, extract_project_links
#from utils import prune_tables

In [ ]:
ordered_cve_columns = ['cve_id', 'published_date', 'last_modified_date', 'description', 'nodes', 'severity',
                       'obtain_all_privilege', 'obtain_user_privilege', 'obtain_other_privilege',
                       'user_interaction_required',
                       'cvss2_vector_string', 'cvss2_access_vector', 'cvss2_access_complexity', 'cvss2_authentication',
                       'cvss2_confidentiality_impact', 'cvss2_integrity_impact', 'cvss2_availability_impact',
                       'cvss2_base_score',
                       'cvss3_vector_string', 'cvss3_attack_vector', 'cvss3_attack_complexity',
                       'cvss3_privileges_required',
                       'cvss3_user_interaction', 'cvss3_scope', 'cvss3_confidentiality_impact',
                       'cvss3_integrity_impact',
                       'cvss3_availability_impact', 'cvss3_base_score', 'cvss3_base_severity',
                       'exploitability_score', 'impact_score', 'ac_insuf_info',
                       'reference_json', 'problemtype_json']

cwe_columns = ['cwe_id', 'cwe_name', 'description', 'extended_description', 'url', 'is_category']

df_cve = pd.DataFrame()

In [ ]:
# Constants
BASE_URL = "https://nvd.nist.gov/feeds/json/cve/1.1/"
DATA_PATH = Path("Data")
CSV_PATH = Path("Data")
START_YEAR = 2002
CURRENT_YEAR = datetime.datetime.now().year

def preprocess_jsons(df_in):
    """
    Flattening CVE_Items and removing the duplicates
    :param df_in: merged dataframe of all years json files
    """
    print('Flattening CVE items and removing the duplicates...')
    cve_items = json_normalize(df_in['CVE_Items'])
    df_cve = pd.concat([df_in.reset_index(), cve_items], axis=1)

    # Removing all CVE entries which have null values in reference-data at [cve.references.reference_data] column
    df_cve = df_cve[df_cve['cve.references.reference_data'].str.len() != 0]

    # Re-ordering and filtering some redundant and unnecessary columns
    df_cve = df_cve.rename(columns={'cve.CVE_data_meta.ID': 'cve_id'})
    df_cve = df_cve.drop(
        labels=[
            'index',
            'CVE_Items',
            'cve.data_type',
            'cve.data_format',
            'cve.data_version',
            'CVE_data_type',
            'CVE_data_format',
            'CVE_data_version',
            'CVE_data_numberOfCVEs',
            'CVE_data_timestamp',
            'cve.CVE_data_meta.ASSIGNER',
            'configurations.CVE_data_version',
            'impact.baseMetricV2.cvssV2.version',
            'impact.baseMetricV2.exploitabilityScore',
            'impact.baseMetricV2.impactScore',
            'impact.baseMetricV3.cvssV3.version',
        ], axis=1, errors='ignore')

    # renaming the column names
    df_cve.columns = [rename_columns(i) for i in df_cve.columns]

    # Check and add columns if they are not present in the dataframe
    for col in ordered_cve_columns:
        if col not in df_cve.columns:
            df_cve[col] = ""

    # ordering the cve columns
    df_cve = df_cve[ordered_cve_columns]

    return df_cve


def rename_columns(name):
    """
    converts the other cases of string to snake_case, and further processing of column names.
    """
    name = name.split('.', 2)[-1].replace('.', '_')
    name = re.sub(r'(?<!^)(?=[A-Z])', '_', name).lower()
    name = name.replace('cvss_v', 'cvss').replace('_data', '_json').replace('description_json', 'description')
    return name

def import_cves():
    """
    gathering CVE records by processing JSON files.
    """
    print('-' * 70)
    for year in range(START_YEAR, CURRENT_YEAR + 1):
        extract_target = 'nvdcve-1.1-' + str(year) + '.json'
        zip_file_url = f"{BASE_URL}nvdcve-1.1-{year}.json.zip"

        # Check if the directory already has the json file or not ?
        if os.path.isfile(Path(DATA_PATH) / 'json' / extract_target):
            #print(f'Reusing the {year} CVE json file that was downloaded earlier...')
            json_file = Path(DATA_PATH) / 'json' / extract_target
        else:
            r = requests.get(zip_file_url)
            z = ZipFile(BytesIO(r.content))  # BytesIO keeps the file in memory
            json_file = z.extract(extract_target, Path(DATA_PATH) / 'json')

        with open(json_file, encoding='utf-8') as f:
            yearly_data = json.load(f)
            if year == START_YEAR:  # initialize the df_methods by the first year data
                df_cve = pd.DataFrame(yearly_data)
            else:
                df_cve = pd.concat([df_cve, pd.DataFrame(yearly_data)], ignore_index=True)
            #print(f'The CVE json for {year} has been merged')

    df_cve = preprocess_jsons(df_cve)
    df_cve = df_cve.applymap(str)
    assert df_cve.cve_id.is_unique, 'Primary keys are not unique in cve records!'
    
    # Save the merged CVE data to CSV
    #df_cve.to_csv('cve.csv', index=False)
    df_cve.to_csv(Path(CSV_PATH) / 'cve_data.csv', index=False)
    print('All CVEs have been merged into the cve table')
    print('CVE data has been saved to cve_data.csv')
    print('-' * 70)

    return df_cve 

In [ ]:
df_cve = import_cves()

In [ ]:
df_cve

In [ ]:
print(df_cve.columns) 

In [ ]:
def assign_cwes_to_cves(df_cve):
    df_cwes = extract_cwe()
    #print(df_cwes)
    # fetching CWE associations to CVE records
    print('Adding CWE category to CVE records...')
    df_cwes_class = df_cve[['cve_id', 'problemtype_json']].copy()
    df_cwes_class['cwe_id'] = add_cwe_class(df_cwes_class['problemtype_json'].tolist())  # list of CWE-IDs' portion

    # exploding the multiple CWEs list of a CVE into multiple rows.
    df_cwes_class = df_cwes_class.assign(
        cwe_id=df_cwes_class.cwe_id).explode('cwe_id').reset_index()[['cve_id', 'cwe_id']]
    df_cwes_class = df_cwes_class.drop_duplicates(subset=['cve_id', 'cwe_id']).reset_index(drop=True)
    df_cwes_class['cwe_id'] = df_cwes_class['cwe_id'].str.replace('unknown', 'NVD-CWE-noinfo')

    no_ref_cwes = set(list(df_cwes_class.cwe_id)).difference(set(list(df_cwes.cwe_id)))
    if len(no_ref_cwes) > 0:
        print('List of CWEs from CVEs that are not associated to cwe table are as follows:')
        print(no_ref_cwes)

    # Applying the assertion to cve-, cwe- and cwe_classification table.
    assert df_cwes.cwe_id.is_unique, "Primary keys are not unique in cwe records!"
    assert df_cwes_class.set_index(['cve_id', 'cwe_id']).index.is_unique, \
        'Primary keys are not unique in cwe_classification records!'
    assert set(list(df_cwes_class.cwe_id)).issubset(set(list(df_cwes.cwe_id))), \
        'Not all foreign keys for the cwe_classification records are present in the cwe table!'

    # Save the data to CSV instead of database
    df_cwes = df_cwes[cwe_columns].reset_index()  # to maintain the order of the columns
    df_cwes.to_csv(Path(CSV_PATH) /'cwe.csv', index=False)
    df_cwes_class.to_csv(Path(CSV_PATH) /'cwe_classification.csv', index=False)
   
    print('Added cwe and cwe_classification tables')
    return df_cwes, df_cwes_class

    
def extract_cwe():
    """
    obtains the table of CWE categories from NVD.nist.gov site
    :return df_CWE: dataframe of CWE category table
    """

    cwe_doc = sorted(Path(DATA_PATH).glob('cwec_*.xml'))
    if len(cwe_doc) > 0:
        print('Reusing the CWE XML file that is already in the directory')
        xtree = et.parse(cwe_doc[-1])
    else:
        cwe_url = 'https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'

        cwe_zip = ZipFile(BytesIO(urlopen(cwe_url).read()))
        cwe_doc = sorted(fnmatch.filter(cwe_zip.namelist(),'cwec_*.xml'))  # assumes all files at top level
        assert len(cwe_doc) > 0, \
            'Cannot find a CWE XML file in https://cwe.mitre.org/data/xml/cwec_latest.xml.zip'
        print(f'Extracting CWE data from {cwe_doc[-1]}')
        cwe_file = cwe_zip.extract(cwe_doc[-1], DATA_PATH)
        xtree = et.parse(cwe_file)
        time.sleep(2)

    xroot = xtree.getroot()
    cat_flag = 0
    rows = []

    # include only types 0, 1 and 2 (0 is for weaknesses, 1 for Categories, 2 for Views, 3 for External_References)
    for parents in xroot[0:2]:
        for node in parents:
            cwe_id = 'CWE-' + str(node.attrib['ID'])
            cwe_name = node.attrib['Name'] if node.attrib['Name'] is not None else None
            description = node[0].text if node[0].text is not None else None
            extended_des = et.tostring(node[1], encoding="unicode", method='text') if cat_flag != 1 else ''
            url = 'https://cwe.mitre.org/data/definitions/' + str(node.attrib['ID']).strip() + '.html' if int(node.attrib['ID']) > 0 else None
            is_cat = True if cat_flag == 1 else False

            rows.append({
                'cwe_id': cwe_id,
                'cwe_name': cwe_name,
                'description': description,
                'extended_description': extended_des,
                'url': url,
                'is_category': is_cat,
            })
        cat_flag += 1

    # explicitly adding three CWEs that are not in the xml file
    rows.append({
        'cwe_id': 'NVD-CWE-noinfo',
        'cwe_name': 'Insufficient Information',
        'description': 'There is insufficient information about the issue to classify it; details are unkown or unspecified.',
        'extended_description': 'Insufficient Information',
        'url': 'https://nvd.nist.gov/vuln/categories',
        'is_category': False
    })
    rows.append({
        'cwe_id': 'NVD-CWE-Other',
        'cwe_name': 'Other',
        'description': 'NVD is only using a subset of CWE for mapping instead of the entire CWE, and the weakness type is not covered by that subset.',
        'extended_description': 'Insufficient Information',
        'url': 'https://nvd.nist.gov/vuln/categories',
        'is_category': False
    })

    df_cwe = pd.DataFrame.from_dict(rows)
    df_cwe = df_cwe.drop_duplicates(subset=['cwe_id']).reset_index(drop=True)
    print( df_cwe.head())
    return df_cwe



def add_cwe_class(problem_col):
    """
    returns CWEs of the CVE.
    """
    cwe_classes = []
    for p in problem_col:
        des = str(p).replace("'", '"')
        des = json.loads(des)
        for cwes in json_normalize(des)["description"]:  # for every cwe of each cve.
            if len(cwes) != 0:
                cwe_classes.append([cwe_id for cwe_id in json_normalize(cwes)["value"]])
            else:
                cwe_classes.append(["unknown"])

    assert len(problem_col) == len(cwe_classes), \
        "Sizes are not equal - Problem occurred while fetching the cwe classification records!"
    return cwe_classes


In [ ]:
df_cwe, cwe_classification=assign_cwes_to_cves(df_cve)

In [ ]:
df_cwe